# **Phase 1: YOLO Training on DAWN (In-domain DAWN -> DAWN)**

This notebook trains and evaluates a YOLO model on the DAWN dataset under the in-domain setting.

Objectives:
Train YOLO on DAWN

*   Evaluate global validation performance
*    Evaluate weather-specific performance on fog, rain, and snow

Experiment summary:
- Model: YOLOv11
- Protocol: In-domain (DAWN → DAWN)
- Dataset: DAWN
- Training set: mixed adverse weather conditions
- Validation set: mixed adverse weather conditions
- Additional evaluation: fog, rain, snow subsets
- Purpose: establish Phase 1 baseline performance

### Connect to Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Install Ultralytics

In [2]:
!pip install -q ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 83.7 MB/s eta 0:00:00


### Import librairies

In [31]:
from ultralytics import YOLO
from pathlib import Path
import json
import os
import time
import cv2
import torch
import pandas as pd


### Clean cache

In [ ]:
!rm -f "/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/labels/train.cache"
!rm -f "/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/labels/val.cache"
!rm -f "/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/labels/test_fog.cache"
!rm -f "/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/labels/test_rain.cache"
!rm -f "/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/labels/test_snow.cache"

### Delete old run

In [ ]:
import shutil
from pathlib import Path

runs_root = Path(
    "/content/drive/MyDrive/Dissertation/Runs/yolo"
)

if runs_root.exists():
    shutil.rmtree(runs_root)

runs_root.mkdir(parents=True, exist_ok=True)

print("Old YOLO runs removed.")

Old YOLO runs removed.


### Check GPU

In [ ]:
!nvidia-smi

Fri May  8 16:52:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             55W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## **Paths and Configuration**

### Define dataset and output path

In [ ]:
# Dataset for YOLO
dataset_root = Path("/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo")

# YAML files
dataset_yaml = dataset_root / "dataset.yaml"
fog_yaml = dataset_root / "fog_only.yaml"
rain_yaml = dataset_root / "rain_only.yaml"
snow_yaml = dataset_root / "snow_only.yaml"

#Output folder for YOLO runs
runs_root = Path("/content/drive/MyDrive/Dissertation/Runs/yolo")

runs_root.mkdir(parents=True, exist_ok=True)

print("Dataset root:", dataset_root)
print("dataset.yaml exists:", dataset_yaml.exists())
print("fog.yaml exists:", fog_yaml.exists())
print("rain.yaml exists:", rain_yaml.exists())
print("snow.yaml exists:", snow_yaml.exists())
print("Runs root:", runs_root)

Dataset root: /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo
dataset.yaml exists: True
fog.yaml exists: True
rain.yaml exists: True
snow.yaml exists: True
Runs root: /content/drive/MyDrive/Dissertation/Runs/yolo


### Check YAML configuration files

In [ ]:
for file_path in [dataset_yaml, fog_yaml, rain_yaml, snow_yaml]:
    print("\n ", file_path.name)
    with open(file_path, "r") as f:
        print(f.read())


  dataset.yaml

path: /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo
train: images/train
val: images/val

nc: 6
names: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


  fog_only.yaml

path: /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo
train: images/train
val: images/test_fog

nc: 6
names: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


  rain_only.yaml

path: /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo
train: images/train
val: images/test_rain

nc: 6
names: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']


  snow_only.yaml

path: /content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo
train: images/train
val: images/test_snow

nc: 6
names: ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck']



In [ ]:
from pathlib import Path

output_root_yolo = Path("/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo")

print("Folder exists:", output_root_yolo.exists())

for file in output_root_yolo.iterdir():
    print(file.name)

Folder exists: True
images
labels
dataset.yaml
fog_only.yaml
rain_only.yaml
snow_only.yaml


# **Model setup**

### Select model variant

In [ ]:
model_name = "yolo11s.pt"
print("Model selected", model_name)

Model selected yolo11s.pt


### Load pre-trained model

In [ ]:
model = YOLO(model_name)
print("YOLO model loaded successfully.")

YOLO model loaded successfully.


# **Training - Phase 1: DAWN -> DAWN**

### Set training hyperparameter

In [ ]:
experiment_name = "dawn_in_domain_yolo11s_seed42"

epochs = 100
imgsz = 640
batch = 16
patience = 10
seed = 42

print("Experiment name:", experiment_name)
print("Epochs:", epochs)
print("Image size:", imgsz)
print("Batch size:", batch)
print("Patience:", patience)
print("Seed:", seed)

Experiment name: dawn_in_domain_yolo11s_seed42
Epochs: 100
Image size: 640
Batch size: 16
Patience: 10
Seed: 42


### Run in-domain training on DAWN

In [ ]:
train_results = model.train(
    data=str(dataset_yaml),
    epochs=epochs,
    imgsz=imgsz,
    batch=batch,
    patience=patience,
    seed=seed,
    project=str(runs_root),
    name=experiment_name,
    pretrained=True,
    verbose=True
)

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=dawn_in_domain_yolo11s_seed42, nbs=64, n

# **Global Evaluation**

In [ ]:
best_model_path = runs_root / experiment_name / "weights" / "best.pt"
last_model_path = runs_root / experiment_name / "weights" / "last.pt"

print("Best model path:", best_model_path)
print("Exists:", best_model_path.exists())

print("Last model path:", last_model_path)
print("Exists:", last_model_path.exists())

Best model path: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/weights/best.pt
Exists: True
Last model path: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/weights/last.pt
Exists: True


In [ ]:
best_model = YOLO(str(best_model_path))
print("Best model reloaded successfully.")

Best model reloaded successfully.


In [ ]:
metrics_global = best_model.val(data=str(dataset_yaml))
print("Global evaluation completed.")

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLO11s summary (fused): 101 layers, 9,415,122 parameters, 0 gradients, 21.3 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.1 ms, read: 68.4±27.4 MB/s, size: 72.0 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/val.cache... 140 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 140/140 53.4Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 9/9 4.1it/s 2.2s
                   all        140       1332      0.578      0.593      0.581       0.34
                person         33         76      0.423      0.589      0.553      0.306
               bicycle          3          3          1      0.577      0.665      0.309
                   car        138       1098      0.666      0.858      0.853      0.555
            motorcycle        

# **Weather-Specific Evaluation**

### Fog evaluation

In [ ]:
metrics_fog = best_model.val(data=str(fog_yaml))
print("Fog evaluation completed.")

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
val: Fast image access ✅ (ping: 0.7±0.2 ms, read: 0.1±0.0 MB/s, size: 48.9 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test_fog... 60 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 60/60 4.5it/s 13.3s
val: New cache created: /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test_fog.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 1.5it/s 2.7s
                   all         60        412      0.805      0.562      0.615      0.385
                person          8         15      0.768        0.8      0.796        0.4
                   car         59        357      0.892      0.765      0.872      0.559
            motorcycle          3          3     

### Rain evaluation

In [ ]:
metrics_rain = best_model.val(data=str(rain_yaml))
print("Rain evaluation completed.")

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
val: Fast image access ✅ (ping: 0.8±0.2 ms, read: 0.1±0.1 MB/s, size: 103.5 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test_rain... 40 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 40/40 4.1it/s 9.8s
val: New cache created: /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test_rain.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.9s/it 5.7s
                   all         40        250      0.618      0.574      0.543      0.357
                person          3          5      0.545        0.4      0.333      0.172
                   car         39        224      0.791      0.763      0.813      0.504
                   bus          2          2   

### Snow evaluation

In [ ]:
metrics_snow = best_model.val(data=str(snow_yaml))
print("Snow evaluation completed.")

Ultralytics 8.4.47 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
val: Fast image access ✅ (ping: 0.7±0.2 ms, read: 0.2±0.1 MB/s, size: 115.1 KB)
val: Scanning /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test_snow... 40 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 40/40 4.0it/s 10.0s
val: New cache created: /content/drive/.shortcut-targets-by-id/17lufoiYQdFrVMwNFeK91HKX-sBA0zN4V/Dissertation/Datasets/processed/dawn_yolo/labels/test_snow.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.2s/it 3.6s
                   all         40        258      0.755      0.603      0.682      0.452
                person          8         13      0.814      0.692      0.759        0.5
                   car         40        224      0.931      0.667      0.837      0.534
                   bus          1          2  

# **Results Extraction**

In [ ]:
def extract_metrics(metrics_obj):
    return {
        "map50": float(metrics_obj.box.map50),
        "map50_95": float(metrics_obj.box.map),
        "mp": float(metrics_obj.box.mp),
        "mr": float(metrics_obj.box.mr)
    }

In [ ]:
results_summary = {
    "global": extract_metrics(metrics_global),
    "fog": extract_metrics(metrics_fog),
    "rain": extract_metrics(metrics_rain),
    "snow": extract_metrics(metrics_snow)
}

results_summary

{'global': {'map50': 0.5812452930219402,
  'map50_95': 0.3396287319883049,
  'mp': 0.5776077486323775,
  'mr': 0.5927543844416351},
 'fog': {'map50': 0.6152619734360698,
  'map50_95': 0.3846645751141019,
  'mp': 0.8054512859364454,
  'mr': 0.5621335015027892},
 'rain': {'map50': 0.5429430571864977,
  'map50_95': 0.3572607295115205,
  'mp': 0.6182433491817393,
  'mr': 0.5737429511278196},
 'snow': {'map50': 0.6822179184938334,
  'map50_95': 0.4520038033502384,
  'mp': 0.7550325595926238,
  'mr': 0.6028949924924568}}

In [ ]:
print("\n YOLO DAWN → DAWN Results Summary:")

for split_name, metrics_dict in results_summary.items():
    print(f"\n--- {split_name.upper()} ---")
    for metric_name, value in metrics_dict.items():
        print(f"{metric_name}: {value:.4f}")


 YOLO DAWN → DAWN Results Summary:

--- GLOBAL ---
map50: 0.5812
map50_95: 0.3396
mp: 0.5776
mr: 0.5928

--- FOG ---
map50: 0.6153
map50_95: 0.3847
mp: 0.8055
mr: 0.5621

--- RAIN ---
map50: 0.5429
map50_95: 0.3573
mp: 0.6182
mr: 0.5737

--- SNOW ---
map50: 0.6822
map50_95: 0.4520
mp: 0.7550
mr: 0.6029


# **Results Saving**

In [ ]:
import json

results_json_path = runs_root / experiment_name / "weather_results_summary.json"

with open(results_json_path, "w") as f:
    json.dump(results_summary, f, indent=4)

print("Results saved to:", results_json_path)

Results saved to: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/weather_results_summary.json


In [ ]:
import pandas as pd

results_csv_path = runs_root / experiment_name / "weather_results_summary.csv"

df = pd.DataFrame(results_summary)
df.to_csv(results_csv_path, index=False)

print("CSV saved to:", results_csv_path)
df

CSV saved to: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/weather_results_summary.csv


,global,fog,rain,snow
map50,0.581245,0.615262,0.542943,0.682218
map50_95,0.339629,0.384665,0.357261,0.452004
mp,0.577608,0.805451,0.618243,0.755033
mr,0.592754,0.562134,0.573743,0.602895


# **Useful Output files**
This section identifies the main output files generated by the experiment, including training curves, confusion matrices, and the best model checkpoint.

In [ ]:
results_png = runs_root / experiment_name / "results.png"
confusion_matrix = runs_root / experiment_name / "confusion_matrix.png"
labels_jpg = runs_root / experiment_name / "labels.jpg"

print("results.png exists:", results_png.exists(), "-", results_png)
print("confusion_matrix.png exists:", confusion_matrix.exists(), "-", confusion_matrix)
print("labels.jpg exists:", labels_jpg.exists(), "-", labels_jpg)
print("best.pt exists:", best_model_path.exists(), "-", best_model_path)

results.png exists: True - /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/results.png
confusion_matrix.png exists: True - /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/confusion_matrix.png
labels.jpg exists: True - /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/labels.jpg
best.pt exists: True - /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/weights/best.pt


# **YOLO FPS Evaluation**

In [5]:
model_path = "/content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/weights/best.pt"

model = YOLO(model_path)

### Fog FPS evaluation

In [11]:
image_dir = "/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/images/test_fog"

image_paths = list(Path(image_dir).glob("*.*"))

images = []

for p in image_paths:
    img = cv2.imread(str(p))
    images.append(img)

print("Loaded images:", len(images))

Loaded images: 60


In [14]:
for _ in range(10):
    _ = model.predict(images[0], verbose=False)

torch.cuda.synchronize()

In [15]:
start = time.time()

for img in images:
    _ = model.predict(img, verbose=False)

torch.cuda.synchronize()

end = time.time()

total_time = end - start

avg_time = total_time / len(images)

fps = 1 / avg_time

print("Total time:", total_time)
print("Average inference time:", avg_time)
print("FPS:", fps)

Total time: 0.7305917739868164
Average inference time: 0.01217652956644694
FPS: 82.12520608134128


### Rain FPS evaluation

In [43]:
image_dir_rain = "/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/images/test_rain"

image_paths_rain = list(Path(image_dir_rain).glob("*.*"))

images_rain = []

for p in image_paths_rain:
    img = cv2.imread(str(p))
    images_rain.append(img)

print("Loaded images:", len(images_rain))

Loaded images: 40


In [44]:
for _ in range(10):
    _ = model.predict(images_rain[0], verbose=False)

torch.cuda.synchronize()

In [45]:
start = time.time()

for img in images_rain:
    _ = model.predict(img, verbose=False)

torch.cuda.synchronize()

end = time.time()

total_time_rain = end - start

avg_time_rain = total_time_rain / len(images_rain)

fps_rain = 1 / avg_time_rain

print("Total time:", total_time_rain)
print("Average inference time:", avg_time_rain)
print("FPS:", fps_rain)

Total time: 0.49079322814941406
Average inference time: 0.012269830703735352
FPS: 81.5007170144219


### Snow FPS evaluation

In [46]:
image_dir_snow = "/content/drive/MyDrive/Dissertation/Datasets/processed/dawn_yolo/images/test_snow"

image_paths_snow = list(Path(image_dir_snow).glob("*.*"))

images_snow = []

for p in image_paths_snow:
    img = cv2.imread(str(p))
    images_snow.append(img)

print("Loaded images:", len(images_snow))

Loaded images: 40


In [47]:
for _ in range(10):
    _ = model.predict(images_snow[0], verbose=False)

torch.cuda.synchronize()

In [48]:
start = time.time()

for img in images_snow:
    _ = model.predict(img, verbose=False)

torch.cuda.synchronize()

end = time.time()

total_time_snow = end - start

avg_time_snow = total_time_snow / len(images_snow)

fps_snow = 1 / avg_time_snow

print("Total time:", total_time_snow)
print("Average inference time:", avg_time_snow)
print("FPS:", fps_snow)

Total time: 0.49833250045776367
Average inference time: 0.012458312511444091
FPS: 80.2676926816059


### Save result

In [49]:
yolo_fps_results = {
    "fog": {
        "total_time": total_time,
        "avg_inference_time": avg_time,
        "FPS": fps
    },

    "rain": {
        "total_time": total_time_rain,
        "avg_inference_time": avg_time_rain,
        "FPS": fps_rain
    },

    "snow": {
        "total_time": total_time_snow,
        "avg_inference_time": avg_time_snow,
        "FPS": fps_snow
    }
}

In [50]:
save_path = Path(
    "/content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/yolo_fps_results.json"
)

with open(save_path, "w") as f:
    json.dump(yolo_fps_results, f, indent=4)

print("YOLO FPS results saved to:", save_path)

YOLO FPS results saved to: /content/drive/MyDrive/Dissertation/Runs/yolo/dawn_in_domain_yolo11s_seed42/yolo_fps_results.json


Preliminary in-domain experiments on the DAWN dataset using YOLOv11 successfully validated the end-to-end training pipeline.
The model was trained for 5 epochs and showed progressive improvement in validation performance.
Final evaluation using the best checkpoint provided a first DAWN → DAWN baseline, while separate fog, rain, and snow evaluations enabled condition-specific analysis.
These preliminary results confirm the feasibility of the proposed methodology and justify full-scale experiments on longer training schedules and additional datasets.


The DAWN → DAWN in-domain evaluation using YOLO11s was successfully completed under the final experimental configuration using an NVIDIA A100 GPU. Training converged after 41 epochs through early stopping (`patience = 10`), with the best checkpoint obtained at epoch 31.

The final model achieved an overall mAP50 of 0.581 and mAP50-95 of 0.340, with weather-specific evaluations showing the strongest performance under snow conditions and lower performance under rain conditions. These results establish the official YOLO11s DAWN in-domain baseline for subsequent cross-dataset and comparative experiments.
